**Etapa 1 — Entender a carteira**

**Objetivo:** descobrir quem são os clientes da base.

**Perguntas:**

Quantos clientes existem?
Qual a distribuição por idade?
Como os clientes estão distribuídos geograficamente?
Existe concentração em determinadas regiões?
Qual a distribuição por gênero?
Qual o saldo médio?
Qual o valor médio das transações?
Qual a frequência média de utilização?

Aqui você ainda não tenta encontrar "oportunidades".

**É o diagnóstico da carteira.**



In [0]:
# Manipulação e estruturação de dados
import pandas as pd
import numpy as np

# Visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning e Métricas (caso vá avançar para modelagem preditiva)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [0]:
df = spark.sql("""
    SELECT *
    FROM bank_transactions
""").toPandas()

df.head()

In [0]:
%sql
select count(*) as total_clientes from bank_transactions


In [0]:
df.info()
df.describe()
df.isnull().sum()

In [0]:
(df.isnull().sum() / len(df) * 100).round(2)

Muito saudável para uma base de mais de 1 milhão de registros.

Alterando os tipos de data para as correta!

In [0]:
###COnvertendo datas

df['CustomerDOB'] = pd.to_datetime(
    df['CustomerDOB'],
    dayfirst=True,
    errors='coerce'
)

df['TransactionDate'] = pd.to_datetime(
    df['TransactionDate'],
    dayfirst=True,
    errors='coerce'
)

### _Ajustando_ "consumerDOB" para uma nova coluna "AGE" para ter uma idade atualizada

In [0]:
%sql
SELECT distinct CustomerDOB
from bank_transactions

In [0]:
# 1. Trata 1800-01-01 como data inválida
df.loc[
    df["CustomerDOB"] == pd.Timestamp("1800-01-01"),
    "CustomerDOB"
] = pd.NaT

# 2. Corrige anos futuros/incompatíveis
df.loc[
    df["CustomerDOB"].dt.year > 2026,
    "CustomerDOB"
] = df.loc[
    df["CustomerDOB"].dt.year > 2026,
    "CustomerDOB"
].apply(lambda x: x.replace(year=x.year - 100))

# 3. Calcula idade
df["Age"] = 2026 - df["CustomerDOB"].dt.year

In [0]:
df["Age"].describe()

In [0]:
print("Menores de 18:", (df["Age"] < 18).sum())
print("18 a 89:", ((df["Age"] >= 18) & (df["Age"] < 90)).sum())
print("90 ou mais:", (df["Age"] >= 90).sum())
print("Nulos:", df["Age"].isna().sum())

In [0]:
df.loc[(df["Age"] < 18) | (df["Age"] > 100), "Age"] = None
df[["CustomerDOB", "Age"]].head(20)

Agrupando "Age " por faixas de idade

In [0]:
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[17, 25, 35, 50, 65, 90, float("inf")],
    labels=["18-25", "26-35", "36-50", "51-65", "66-90", "90+"],
    include_lowest=True
)

In [0]:
df["AgeGroup"].isna().sum()

In [0]:
df["AgeGroup"]. describe()

In [0]:
df_fora_regra = df[
    (df["Age"] < 18) |
    (df["Age"] >= 90) |
    (df["Age"].isna())
]

print(df_fora_regra[["CustomerDOB", "Age"]])

In [0]:
print(df_fora_regra.shape[0])

In [0]:
df["AgeGroup"].value_counts().sort_index()

In [0]:
age_pct = (
    df["AgeGroup"]
    .value_counts(normalize=True)
    .reindex(["18-25", "26-35", "36-50", "51-65", "65+"])
    * 100
)

plt.figure(figsize=(10, 6))

ax = age_pct.plot(kind="bar")

plt.title("Distribuição dos Clientes por Faixa Etária")
plt.xlabel("Faixa Etária")
plt.ylabel("Percentual (%)")
plt.xticks(rotation=0)

for i, value in enumerate(age_pct):
    ax.text(
        i,
        value,
        f"{value:.1f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [0]:
df["Age"].describe()

**Como os clientes estão distribuídos geograficamente?**

In [0]:
CustLocation_pct = (
    df["CustLocation"]
    .value_counts(normalize=True)
    .head(10)
    * 100
)

plt.figure(figsize=(12, 6))

ax = CustLocation_pct.plot(kind="bar")

plt.title("Top 10 Localizações dos Clientes")
plt.xlabel("Localização")
plt.ylabel("Percentual (%)")
plt.xticks(rotation=45, ha="right")

for i, value in enumerate(CustLocation_pct):
    ax.text(
        i,
        value,
        f"{value:.1f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [0]:
%sql
select CustLocation,
COUNT(*) AS total_clientes,
ROUND((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()), 2) AS percentual_total
from bank_transactions
group by CustLocation
order by total_clientes desc limit 10





**Existe concentração em determinadas regiões?**

In [0]:
import plotly.express as px

# Coordenadas aproximadas das principais cidades
coords = {
    "MUMBAI": [19.0760, 72.8777],
    "NEW DELHI": [28.6139, 77.2090],
    "BANGALORE": [12.9716, 77.5946],
    "GURGAON": [28.4595, 77.0266],
    "DELHI": [28.7041, 77.1025],
    "NOIDA": [28.5355, 77.3910],
    "CHENNAI": [13.0827, 80.2707],
    "PUNE": [18.5204, 73.8567],
    "HYDERABAD": [17.3850, 78.4867],
    "THANE": [19.2183, 72.9781]
}

location_df = CustLocation_pct.reset_index()
location_df.columns = ["Location", "Percentage"]

location_df["Latitude"] = location_df["Location"].map(
    lambda x: coords.get(x, [None, None])[0]
)

location_df["Longitude"] = location_df["Location"].map(
    lambda x: coords.get(x, [None, None])[1]
)

fig = px.scatter_geo(
    location_df.dropna(),
    lat="Latitude",
    lon="Longitude",
    size="Percentage",
    hover_name="Location",
    hover_data={"Percentage": ":.2f"},
    scope="asia",
    title="Concentração Geográfica dos Clientes"
)

fig.show()

As cinco principais localidades, juntas, representam aproximadamente **39,6% de toda a base de clientes**.

A distribuição é **bastante concentrada nos grandes centros urbanos da Índia**. Mumbai lidera com **9,88%** dos clientes, seguida por New Delhi (**8,10%**), Bangalore (**7,78%**), Gurgaon (**7,04%**) e Delhi (**6,77%**).

Além disso, aparecem outros importantes polos urbanos, como Noida, Chennai, Pune e Hyderabad, mas com participações individuais menores.

In [0]:
df["CustLocation"].describe()


**Sim, existe uma concentração geográfica relevante.**

O principal destaque é o eixo **Delhi–NCR**, considerando **New Delhi, Gurgaon, Delhi, Noida e Ghaziabad**. Somando essas localidades, temos aproximadamente **26,6% da base**.

Também há uma concentração importante no entorno de **Mumbai**, considerando Mumbai, Thane e Navi Mumbai, que representam aproximadamente **13,2%**.

Isso indica que a carteira não está distribuída de maneira uniforme pelo território: existe forte presença em **clusters metropolitanos**, especialmente nas regiões de **Delhi-NCR e Mumbai**.

**Qual a distribuição por gênero?**

In [0]:
%sql
select CustGender,
COUNT(*) AS total_clientes,
ROUND((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()), 2) AS percentual_total
from bank_transactions
group by CustGender
order by total_clientes desc 



In [0]:
df['CustGender'].value_counts().sort_index()

In [0]:
CustGender_pct = (
    df["CustGender"]
    .value_counts(normalize=True)
    * 100
)

plt.figure(figsize=(8, 6))
ax = CustGender_pct.plot(kind="bar")

plt.title("Distribuição dos Clientes por Gênero")
plt.xlabel("Gênero")
plt.ylabel("Percentual (%)")
plt.xticks(rotation=0)

for i, value in enumerate(CustGender_pct):
    ax.text(
        i,
        value,
        f"{value:.1f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

A carteira de clientes é **predominantemente masculina**, com aproximadamente **3 em cada 4 clientes sendo homens**. As mulheres representam pouco mais de um quarto da base.

Para uma análise de negócio, essa concentração pode ser relevante para avaliar se **produtos, campanhas e estratégias de relacionamento estão adequadamente direcionados aos diferentes perfis de clientes**.

**Qual o saldo médio**

In [0]:
df["CustAccountBalance"].describe()

**A base é extremamente assimétrica**

A média (115,4 mil) é quase 7x maior que a mediana (16,8 mil).
**Isso mostra que poucos clientes possuem saldos muito elevados e estão puxando a média para cima.**

O cliente "típico" tem saldo bem menor
25% dos clientes: até R$ 4,7 mil
**50%: até R$ 16,8 mil**
75%: até R$ 57,7 mil
25% estão acima de R$ 57,7 mil

**Então, para representar o comportamento típico da base, R$ 16,8 mil é muito mais representativo que R$ 115,4 mil.**

"Os 10% dos clientes com maior saldo concentram quantos % de todo o dinheiro da base?"

In [0]:
df["CustAccountBalance"].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99])

In [0]:
df["CustAccountBalance"].nlargest(
    int(len(df) * 0.10)
).sum() / df["CustAccountBalance"].sum() * 100

In [0]:
df["CustAccountBalance"].nlargest(
    int(len(df) * 0.01)
).sum() / df["CustAccountBalance"].sum() * 100

In [0]:
df["CustAccountBalance"].nlargest(
    int(len(df) * 0.05)
).sum() / df["CustAccountBalance"].sum() * 100

In [0]:
df["CustAccountBalance"].nlargest(
    int(len(df) * 0.20)
).sum() / df["CustAccountBalance"].sum() * 100

**A base apresenta elevada concentração de saldo: os 10% de clientes com maiores saldos concentram aproximadamente 76,8% do saldo total, enquanto os 80% de menor saldo concentram apenas 12,3%.**

**Saída**

**Um perfil geral:**

"A carteira é predominantemente composta por X perfil de clientes, concentrados em X regiões, com ticket médio de X e frequência média de X transações."